# Programming Assignment 2: Statistics, Optimisation and Learning on manifolds

**DD2603 Geometric Robot Learning, take-home assignment 2.**

Programming Assignment 1 gave you the geometry of a few fixed manifolds: spheres, rotations, robot poses. This assignment uses that geometry in two ways.
* **Statistics** (Parts 1–2): what is the mean of a set of points on a sphere, and what is a Gaussian on a sphere?
* **Learning a geometry from data** (Parts 3–5): demonstrations of a robot motion define a Riemannian metric on the workspace whose shortest paths, the geodesics, look like the demonstrations. The metric is a field of symmetric positive definite (SPD) matrices, and the SPD matrices are themselves a manifold, on which you will run the same kind of gradient descent as for the mean of rotations.

| Part | Topic | Data | Points |
|---|---|---|---|
| 1 | The Fréchet mean | synthetic | 14 |
| 2 | The wrapped Gaussian | synthetic | 14 |
| 3 | Metrics from data density | pouring | 12 |
| 4 | Learning a metric on the SPD manifold | pouring | 34 |
| 5 | An SPD network for the metric | pouring | 26 |

### What you should be able to do after this assignment
1. Compute a Fréchet mean by Riemannian gradient descent, and explain how the step size and the curvature set its convergence.
2. Sample from a wrapped Gaussian, and explain how it differs from a Gaussian in the plane.
3. Explain what a Riemannian metric field on the workspace is, how its geodesics are computed, and how it can be built from data.
4. Optimise parameters that live on the SPD manifold with Riemannian gradient descent, and compare with Euclidean workarounds.
5. Build and train a network whose output is an SPD matrix, and compare Riemannian and Euclidean training of its weights.

## How the assignment works

The structure is the same as in Programming Assignment 1:

1. **Context:** a short recap of the theory, with the formulas you need.
2. **Tasks:** a numbered list naming each function to write, with input and output shapes.
3. **Skeleton:** code cells with signatures and docstrings. Replace each `raise NotImplementedError` with your code. Do not rename functions or change signatures: the checks call them directly.
4. **Public checks:** a provided cell, `checks.partN(...)`, that tests your code and prints `PASS`/`FAIL`/`TODO` per function. Run it as often as you like.
5. **Experiments:** the experiment code is provided: data loading, training loops, tables and figures. It calls *your* functions, so it runs once your implementation is complete. Run these cells and use the numbers and figures they produce in your answers.
6. **Questions:** each question has an answer cell directly below it, starting with **Answer Qx.y**. Write your answer there.

### Rules
* **Your own functions must not use geomstats or geoopt.** Both are *oracles* to compare against, used only in cells that start with `# ORACLE` and inside provided code (Part 5 trains with geoopt's optimiser).
* Parts 1–2 use NumPy. Parts 3–5 use **PyTorch**, because they need derivatives of derivatives, which autograd provides. You need only tensors, matrix operations and `torch.autograd.grad`; the context says where. All tensors are `float64`.
* Generative AI tools: follow the course policy. Answers must be tied to *your own* output (see below).

### What a full-mark written answer contains
1. The claim, in one or two sentences.
2. **A number or figure from your own run** that supports it.
3. Where relevant, the reason in terms of the geometry (which manifold, which metric, which map).

An answer that is correct but not tied to your own output gets half marks.

### Grading and submission
Each part is graded on your implementation (the public checks pass and the code is readable), the figures and tables of the experiments (they must be in your executed notebook), and the written answers. Submit this notebook, executed, with all outputs (`.ipynb`). Before submitting, run *Runtime → Restart session and run all* and make sure that every cell runs without errors. Parts 4 and 5 train models: the whole notebook takes about 10 minutes on Colab's CPU (no GPU needed).

## The provided code: `assignment2utils`

The first code cell installs `assignment2utils`, with the data, helpers and public checks. Everything is documented: run `help(mt)` or `lp.plot_metric_ellipses?` to see the docstrings.

* **`data`**: `load_pouring_paths()` (Parts 3–5).
* **`manifolds`** (Parts 1–2): the solution of Programming Assignment 1, `Sphere`, `UnitQuaternions`, `PoseManifold`, and `projected_normal`, a simple sampler for test data.
* **`rotations`** (Parts 1–2): conversions with quaternions in `(w, x, y, z)` order, as in Programming Assignment 1.
* **`plotting`**: the sphere and convergence plots of Programming Assignment 1, plus `path_axes`, `plot_path`, `plot_paths_3d`, `plot_metric`, `plot_anchors` and `plot_curves` for Parts 3–5.
* **`metrics`** (Parts 3–5, PyTorch):
  * SPD matrix functions with safe gradients: `spd_log`, `spd_sqrtm`, `spd_invsqrtm`, `spd_dist`, and `sym`, `vec_to_sym`;
  * geodesics: `geodesic_residual`, `geodesic_bvp`, `christoffel_fd` (a slow reference), `euclidean` (the metric $M=I$);
  * evaluation: `evaluate`, `path_error_mm`, `resample`, `tensors`.
* **`checks`**: `part1(...)` to `part5(...)`, the public checks.
* **`oracle`**: geomstats. **Only in `# ORACLE` cells.**

### The dataset
* **`load_pouring_paths()`**: the bottle positions of the 10 pouring demos of Programming Assignment 1 (5 glass positions × water, wine), cut to the approach segment and sampled at 100 times: `x`, `v`, `a (10, 100, 3)` (position, velocity, acceleration), `liquid`, `position (10,)`, the whole recorded path `full (10, 480, 3)`; the training set `train_x`, `train_v`, `train_a (20, 100, 3)` (10 curves to the glasses on the left, 10 to the glasses on the right) and the indices `test` of the 2 held-out demos. Part 3 explains it.

### Your API: what you implement

| Part | You implement |
|---|---|
| 1 | `frechet_mean` |
| 2 | `WrappedGaussian` |
| 3 | `ConformalMetric` |
| 4 | `christoffel`, `airm_rgrad`, `airm_exp`, `rgd_step` |
| 5 | `bimap`, `reeig`, the manifold parameters and `shape` of `SPDMetricNet`, the optimiser in `fit_net` |

Parts 1 and 2 are independent of Parts 3–5. Within Parts 3–5, later parts use your earlier functions: Part 5 uses `christoffel` and `airm_exp` from Part 4.

## Conventions

* **Parts 1–2** follow Programming Assignment 1: quaternions are `(w, x, y, z)`, the base point comes first (`S.log(p, q)` is $\mathrm{Log}_p(q)$), angles are radians inside the API and degrees in printed tables.
* **Parts 3–5**: positions are in metres, and a batch of positions is a tensor of shape `(N, 3)`. A **metric field** is any function `metric(x)` that returns a tensor `(N, 3, 3)` of SPD matrices, written with torch operations so that autograd can differentiate it with respect to `x`. Demos are tensors `(D, T, 3)`: D demos, T times.

Run the next cell first, and run it again every time Colab restarts the runtime. It installs `assignment2utils` (with the data), geomstats and geoopt if they are missing, and prints the versions.

In [ ]:
# Part 0 setup: safe to re-run.
GITHUB_REPO = "GeoRob-Lab/dd2603-course"   # the course repository on GitHub (set by the TA)
ASSIGNMENT_RELEASE = "assignment2-v1.0"                 # the release of this assignment (a git tag)

import importlib.util
if importlib.util.find_spec("assignment2utils") is None:     # not installed yet, e.g. a fresh Colab runtime
    %pip install -q "assignment2utils @ git+https://github.com/{GITHUB_REPO}@{ASSIGNMENT_RELEASE}#subdirectory=assignments/assignment2"

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
import assignment2utils
from assignment2utils import data, checks
from assignment2utils import rotations as rot
from assignment2utils import plotting as lp
from assignment2utils import metrics as mt

warnings.filterwarnings("ignore", message=".*torch.jit.*")    # a deprecation warning printed by geoopt
torch.set_default_dtype(torch.float64)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=4, suppress=True)
assignment2utils.print_versions()

All randomness in the assignment goes through `rng` (NumPy) and the seeds of the PyTorch models. You may change `SEED`; your synthetic data, and therefore some of your numbers, depend on it.

In [ ]:
SEED = 2603
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED);

### Prerequisites from Programming Assignment 1 (provided)
`assignment2utils.manifolds` contains the solution of Programming Assignment 1: `Sphere`, `UnitQuaternions` and `PoseManifold`, which the Fréchet mean and the wrapped Gaussian build on (run `help(Sphere)` to see the methods). It also provides `projected_normal`, a simple way to make random points near a given point of a sphere, which Part 1 uses as test data.

In [ ]:
from assignment2utils.manifolds import Sphere, UnitQuaternions, PoseManifold, projected_normal

S2 = Sphere(2)
Q = UnitQuaternions()

---
# Part 1: The Fréchet mean (14 points)

### Context
In $\mathbb R^n$ the mean minimises the sum of squared distances. On a manifold, take that as the *definition*: the **Fréchet mean** of $x_1,\dots,x_N$ is
$$\bar x=\arg\min_{m}\;F(m),\qquad F(m)=\frac1{2N}\sum_{i=1}^N d(m,x_i)^2 .$$
Away from the cut locus, the Riemannian gradient of $\frac12d(m,x)^2$ is $-\mathrm{Log}_m x$, so
$$\operatorname{grad}F(m)=-\frac1N\sum_i\mathrm{Log}_m x_i .$$
**Riemannian gradient descent** takes a step against the gradient in the tangent space and maps it back with Exp:
$$m_{k+1}=\mathrm{Exp}_{m_k}\Bigl(\alpha\cdot\frac1N\sum_i\mathrm{Log}_{m_k}x_i\Bigr).$$
With $\alpha=1$ this is "average the Log vectors, then step to their mean", the classical Karcher iteration. The mean is unique if the data lies in a geodesic ball of radius smaller than π/2 on the unit sphere.

The experiments use test data from `projected_normal` (see the prerequisites).

### Tasks
* **1.1** `frechet_mean(manifold, X, init=None, step=1.0, tol=1e-10, max_iter=200)`. Start from `init` (default `X[0]`) and iterate until $\lVert\operatorname{grad}F\rVert<$ `tol`. Return the mean and a `history` dict with the lists `"objective"` (the value of $F$) and `"grad_norm"` (the Riemannian norm of the gradient), one entry per iterate, starting with the initial point. It must work unchanged for `Sphere`, `UnitQuaternions` and `PoseManifold`: use only `exp`, `log`, `dist` and `norm`.

In [ ]:
def frechet_mean(manifold, X, init=None, step=1.0, tol=1e-10, max_iter=200):
    """Riemannian gradient descent for the Fréchet mean of the points X (N, d)."""
    X = np.asarray(X, dtype=float)
    m = X[0].copy() if init is None else np.asarray(init, dtype=float).copy()
    history = {"objective": [], "grad_norm": []}
    raise NotImplementedError


checks.part1(Sphere, PoseManifold, frechet_mean)

### Experiment 1.1: convergence and step size
For 200 points from `projected_normal` around μ with covariance $0.3^2I$, the provided cell runs your `frechet_mean` with step sizes $\alpha\in\{0.25,0.5,1,1.5,2\}$ and plots the gradient norm against the iteration, one curve per α, on a log scale (`lp.plot_convergence`). The cell below the plot prints the contraction factor $\lVert\operatorname{grad}F(m_{k+1})\rVert/\lVert\operatorname{grad}F(m_k)\rVert$ per step.

→ Q1.2.

In [ ]:
mu1 = S2.projection(np.array([0.3, 0.2, 1.0]))
X1 = projected_normal(mu1, 0.3**2 * np.eye(3), 200, rng)
hists = {}
for alpha in [0.25, 0.5, 1.0, 1.5, 2.0]:
    _, hists[f"α = {alpha}"] = frechet_mean(S2, X1, step=alpha, max_iter=60, tol=1e-14)
lp.plot_convergence(hists, key="grad_norm", title="Gradient norm, σ = 0.3, N = 200")

plt.show()

In [ ]:
for k, h in hists.items():
    g = np.asarray(h["grad_norm"])
    rate = np.median(g[2:8] / g[1:7]) if len(g) > 8 else np.nan
    print(f"{k:8s} iterations: {len(g) - 1:3d}, final grad norm {g[-1]:.1e}, contraction per step ≈ {rate:.3f}")

### Experiment 1.2: initialisation and non-uniqueness
(a) For a sample spread over the whole sphere (`projected_normal` with covariance $2.5^2I$), the provided cell runs `frechet_mean` from 8 random initial points and prints the limits and their distance from the generating point μ.

(b) A bimodal sample: an anisotropic cluster at the north pole and its mirror image at the south pole. The code makes the sample exactly symmetric under a half-turn about the z-axis and under the mirror $z\to-z$, so that every minimiser $m$ of $F$ has a symmetric twin. The cell runs `frechet_mean` from the same 8 initial points and prints the limits and the final objective.

The figure shows each sample (grey) and the limits from the 8 initial points (stars).

→ Q1.1.

In [ ]:
init_pts = S2.random_point(8, rng=rng)
X_wide = projected_normal(mu1, 2.5**2 * np.eye(3), 500, rng)
# Provided: the symmetric bimodal sample
X_n = projected_normal(np.array([0, 0, 1.0]), np.diag([0.3, 0.05, 0.0]) ** 2, 25, rng)
X_n = np.concatenate([X_n, X_n * [-1, -1, 1]])          # + half-turn about z
X_bi = np.concatenate([X_n, X_n * [1, 1, -1]])           # + mirror z -> -z
fig = plt.figure(figsize=(12, 5))
for j, (name, Xd) in enumerate([("spread, σ = 2.5", X_wide), ("bimodal, poles", X_bi)]):
    ax = fig.add_subplot(1, 2, j + 1, projection="3d")
    lp.sphere_axes(ax, title=name)
    lp.plot_on_sphere(Xd, ax=ax, s=3, color="0.5")
    print(name)
    for i, m0 in enumerate(init_pts):
        m, h = frechet_mean(S2, Xd, init=m0, max_iter=500)
        ax.scatter(*m, color=f"C{i}", s=60, marker="*", depthshade=False)
        print(f"  init {i}: mean = {m.round(3)}, F = {h['objective'][-1]:.5f}, "
              f"d(mean, μ) = {np.degrees(S2.dist(m, mu1)):5.1f} deg, iterations = {len(h['objective']) - 1}")
plt.tight_layout()
plt.show()

### Questions, Part 1
**Q1.1** The context gives a condition under which the Fréchet mean on the unit sphere is unique. Do the two samples of Experiment 1.2 satisfy it? What do your runs from 8 initial points show about uniqueness for each sample, and does that contradict the condition?

**Answer Q1.1:**

_Your answer here._

**Q1.2** In $\mathbb R^n$, where $\mathrm{Log}_m x=x-m$ and $\mathrm{Exp}_m v=m+v$, write the iteration of the context in closed form and show that the error $m_k-\bar x$ to the arithmetic mean $\bar x$ is multiplied by $1-\alpha$ at every step. For which α does it converge, and what happens for α = 1 and α = 2? Compare this prediction with the contraction factors printed in Experiment 1.1. Why does α = 1 not reach the mean in one step on the sphere, and why does α = 2 still converge there?


**Answer Q1.2:**

_Your answer here._

**Checkpoint, Part 1.** Public checks pass; the convergence plot; the initialisation figure; Q1.1 and Q1.2 answered.

---
# Part 2: The wrapped Gaussian and sampling (14 points)

### Context
There is no "Gaussian on the sphere" with all the nice properties of the Euclidean one, but there is a simple construction that is used in practice (for example as the noise model of a diffusion policy on rotations): **sample a Gaussian in the tangent space at the mean and map it to the manifold with Exp.**

1. Choose an orthonormal basis $B=[b_1,\dots,b_n]\in\mathbb R^{(n+1)\times n}$ of $T_\mu S^n$, so that tangent vectors have coordinates $z\in\mathbb R^n$.
2. Sample $z\sim\mathcal N(0,\Sigma)$ in $\mathbb R^n$ and set $v=Bz\in T_\mu S^n$.
3. Return $x=\mathrm{Exp}_\mu(v)$.

The result is the **wrapped Gaussian** $\mathcal{WN}(\mu,\Sigma)$, the push-forward of the tangent Gaussian by $\mathrm{Exp}_\mu$. Two things make it differ from a Gaussian in $\mathbb R^n$:
* **Jacobian.** $\mathrm{Exp}_\mu$ squeezes area: a thin ring of radius $r$ in $T_\mu S^2$ has circumference $2\pi r$, but its image on $S^2$ has circumference $2\pi\sin r$. The density of $x$ with respect to the area on the sphere therefore contains the factor $r/\sin r$ (Q2.1).
* **Wrapping.** For $\lVert v\rVert>\pi$ the sample passes the cut locus and comes back from the other side: $\mathrm{Log}_\mu(x)\neq v$, and its mass adds to the density at a nearer point.

### Tasks
* **2.1** `WrappedGaussian.tangent_basis()`: an orthonormal basis of $T_\mu S^n$, shape `(n+1, n)`, unless a basis was given to the constructor. *Hint:* the SVD of the `1 × (n+1)` matrix `mean[None]`.
* **2.2** `tangent_samples(n_samples, rng)`: the tangent vectors $v=Bz$, shape `(n_samples, n+1)`.
* **2.3** `sample(n_samples, rng, return_tangent=False)`: the points $x=\mathrm{Exp}_\mu(v)$, and also $v$ if `return_tangent`.

In [ ]:
class WrappedGaussian:
    """
    Wrapped Gaussian on a sphere: z ~ N(0, cov) in R^n, v = B z in T_mean, x = Exp_mean(v).
    manifold: a Sphere(n) or UnitQuaternions(); mean: (n+1,); cov: (n, n) in the coordinates of B;
    basis: optional (n+1, n) orthonormal basis of the tangent space at mean.
    """

    def __init__(self, manifold, mean, cov, basis=None):
        self.manifold = manifold
        self.mean = np.asarray(mean, dtype=float)
        self.cov = np.atleast_2d(np.asarray(cov, dtype=float))
        self.basis = None if basis is None else np.asarray(basis, dtype=float)

    def __repr__(self):
        return f"WrappedGaussian({self.manifold}, mean={self.mean.round(3)}, cov diag={np.diag(self.cov).round(4)})"

    def tangent_basis(self):
        """
        Orthonormal basis of the tangent space at the mean, as the columns of B, shape (n+1, n).
        The columns have unit length, are orthogonal to each other and to `mean`, so that
        v = B z is tangent at the mean for every z in R^n.
        If a basis was given to the constructor, return it unchanged.
        Hint: the right singular vectors of the 1 x (n+1) matrix mean[None].
        """
        if self.basis is not None:
            return self.basis
        raise NotImplementedError

    def tangent_samples(self, n_samples, rng=None):
        """
        Tangent vectors at the mean: draw z_1, ..., z_N ~ N(0, cov) in R^n
        (rng.multivariate_normal) and return V with rows v_i = B z_i, shape (n_samples, n+1).
        Every row is tangent at the mean.
        """
        rng = np.random.default_rng() if rng is None else rng
        raise NotImplementedError

    def sample(self, n_samples, rng=None, return_tangent=False):
        """
        Points on the manifold: X with rows x_i = Exp_mean(v_i) for the tangent samples v_i,
        shape (n_samples, n+1). If return_tangent is True, return (X, V), where V are the
        tangent vectors that generated X (needed to see which samples wrapped around).
        """
        raise NotImplementedError


checks.part2(Sphere, UnitQuaternions, WrappedGaussian)

### Experiment 2.1: from concentrated to wrapped
A sample wraps around the sphere exactly when its tangent vector is longer than π, the distance to the cut locus. We therefore choose σ as fractions of π: for μ the north pole and isotropic $\Sigma=\sigma^2I$ with $\sigma\in\{\pi/16,\ \pi/8,\ \pi/4,\ \pi/2,\ \pi\}$, the provided cell draws 2000 samples each from your `WrappedGaussian` and plots a grid with one column per σ:
* top row: the samples on the sphere;
* bottom row: the histograms of the tangent lengths $\lVert v\rVert/\pi$ and of the distances $d(\mu,x)/\pi$ on the same axes, with a vertical line at 1 (the cut locus).

Compare the two histograms in each column: where do they agree, and where does the mass of $\lVert v\rVert$ beyond 1 end up?

→ Q2.2.

In [ ]:
mu = np.array([0.0, 0.0, 1.0])
sigmas = np.pi * np.array([1 / 16, 1 / 8, 1 / 4, 1 / 2, 1])
sigma_names = ["π/16", "π/8", "π/4", "π/2", "π"]
fig = plt.figure(figsize=(18, 7))
for j, (s, name) in enumerate(zip(sigmas, sigma_names)):
    X, V = WrappedGaussian(S2, mu, s**2 * np.eye(2)).sample(2000, rng=rng, return_tangent=True)
    ax = fig.add_subplot(2, 5, j + 1, projection="3d")
    lp.sphere_axes(ax, elev=10, title=f"σ = {name}")
    lp.plot_on_sphere(X, ax=ax, s=2, color="C0")
    ax = fig.add_subplot(2, 5, j + 6)
    bins = np.linspace(0, max(1.0, np.linalg.norm(V, axis=-1).max() / np.pi), 40)
    ax.hist(np.linalg.norm(V, axis=-1) / np.pi, bins=bins, alpha=0.5, density=True, label="|v| / π")
    ax.hist(S2.dist(mu, X) / np.pi, bins=bins, alpha=0.5, density=True, label="d(μ, x) / π")
    ax.axvline(1.0, color="k", ls=":")
    ax.set_xlabel("distance / π"); ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### Experiment 2.2: wrapped versus intrinsic Gaussian
The wrapped Gaussian is defined by how it is *sampled*: a Gaussian in the tangent space, pushed through Exp. Another common definition starts from the *density* instead. The **intrinsic (Riemannian) normal** has density $\propto\exp\bigl(-d(\mu,x)^2/(2\sigma^2)\bigr)$ with respect to the area on the sphere: the Euclidean formula, with the Euclidean distance replaced by the geodesic one. Its density is simple, but sampling from it needs rejection sampling, so we take its samples from geomstats (`random_riemannian_normal`).

Both use the same μ, the same σ and the same distance, but they are different distributions. A convenient way to compare them is the distribution of the distance $r=d(\mu,x)$: the probability of a thin ring of radius $r$ around μ is the density on the ring times the ring's length. On the sphere that length is $2\pi\sin r$; in the tangent plane it is $2\pi r$. For $r<\pi$ this gives
$$p_\text{intrinsic}(r)\propto\sin r\;e^{-r^2/2\sigma^2},\qquad p_\text{wrapped}(r)\propto r\;e^{-r^2/2\sigma^2}\ \ (\text{plus the mass that wrapped around from }\lVert v\rVert>\pi).$$

The provided ORACLE cell plots, for σ = 0.2 and σ = 1.5, the histograms of $d(\mu,x)$ for both distributions together with these two curves, and prints the mean distance from μ. This is a contrast, not an oracle check: the two are not supposed to agree.

→ Q2.1 and Q2.3.

In [ ]:
# ORACLE
from assignment2utils import oracle
G = oracle.Hypersphere(dim=2)
np.random.seed(SEED)                   # geomstats samples with NumPy's global generator
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, s in zip(axs, [0.2, 1.5]):
    Xw = WrappedGaussian(S2, mu, s**2 * np.eye(2)).sample(20000, rng=rng)
    Xi = G.random_riemannian_normal(mean=mu, precision=1 / s**2, n_samples=20000)
    r = np.linspace(1e-3, np.pi, 300)
    bins = np.linspace(0, np.pi, 50)
    ax.hist(S2.dist(mu, Xw), bins=bins, density=True, alpha=0.45, label="wrapped (yours)")
    ax.hist(S2.dist(mu, Xi), bins=bins, density=True, alpha=0.45, label="intrinsic (geomstats)")
    for f, c, name in [(r, "C0", "∝ r exp(-r²/2σ²)"), (np.sin(r), "C1", "∝ sin r exp(-r²/2σ²)")]:
        dens = f * np.exp(-r**2 / (2 * s**2))
        ax.plot(r, dens / np.trapezoid(dens, r), color=c, lw=2, label=name)
    ax.set_title(f"σ = {s} rad"); ax.set_xlabel("d(μ, x) [rad]"); ax.legend(fontsize=7)
    print(f"σ = {s}: mean distance wrapped {S2.dist(mu, Xw).mean():.3f}, intrinsic {S2.dist(mu, Xi).mean():.3f} rad")
plt.tight_layout(); plt.show()

### Questions, Part 2
**Q2.1** Derive the density of the wrapped Gaussian on $S^2$, with respect to the area on the sphere, at a point $x$ with $r=\lVert\mathrm{Log}_\mu x\rVert<\pi$, ignoring the wrapped mass. Where does it show in Experiment 2.2?

**Answer Q2.1:**

_Your answer here._

**Q2.2** Is the generating point μ the Fréchet mean of the samples? Answer for symmetric Σ and small σ, and say what changes for large σ. (You can check it with your `frechet_mean`.)

**Answer Q2.2:**

_Your answer here._

**Q2.3** In Experiment 2.2, how do the samples of the wrapped and of the intrinsic Gaussian differ, for small σ and for large σ? Explain the difference from the way each distribution is defined.

**Answer Q2.3:**

_Your answer here._

**Checkpoint, Part 2.** Public checks pass; two figures; Q2.1 to Q2.3 answered.

---
# Part 3: Metrics from data density (12 points)

### Context
A **Riemannian metric field** on the workspace assigns to every position $x\in\mathbb R^3$ an SPD matrix $M(x)$ that measures velocities: moving with velocity $\dot x$ at $x$ has speed $\lVert\dot x\rVert_{M(x)}=\sqrt{\dot x^\top M(x)\,\dot x}$. The energy of a path $x(t)$, $t\in[0,1]$, is
$$E=\int_0^1\dot x^\top M(x)\,\dot x\;dt ,$$
and the **geodesic** between two points is the path of least energy (it is also the shortest path, travelled at constant speed). For $M=I$ the geodesics are straight lines. Where $M(x)$ is large, moving is expensive and geodesics avoid the region; where it is small, they are drawn in. An anisotropic $M(x)$ makes some directions cheaper than others.

A metric field is therefore a **motion model**: to move from $x_0$ to $x_1$, follow the geodesic. Parts 3 to 5 ask how to obtain $M$ from demonstrations, so that the geodesics look like the demonstrations, also between a start and a goal that were not demonstrated.

**The data.** Bottle paths from the pouring demos of Programming Assignment 1, positions only: the **approach** to the glass, from the moment the bottle starts to move until it slows down at the glass. The **training set** has 20 curves in two separate "lanes": 10 to glasses on the left and 10 to glasses on the right; none goes to the centre. Two more demos, one per side (wine 2 and wine 4), are **held out for testing**.

**Evaluation.** Compute the geodesic from the start of a demo to its end and compare it with the demo: the **path error** is the mean distance between the two paths, in mm, after both are resampled to 100 points equally spaced in arc length. We report it for the 2 held-out demos and, averaged, for 6 of the training curves (the recorded demos among them). The straight line is the baseline. Geodesics are computed by `mt.geodesic_bvp`, which minimises the energy of a discretised path with 64 points; `mt.evaluate(metric, paths, idx)` does this for a list of real demos.

**Density metrics.** The simplest data-driven metric makes motion cheap where there is data. With the kernel density estimate of the training positions $x_1,\dots,x_N$,
$$\hat p(x)=\frac1N\sum_{n=1}^N\exp\Bigl(-\frac{\lVert x-x_n\rVert^2}{2\sigma^2}\Bigr),$$
the **conformal density metric** is $M(x)=\dfrac{1}{\hat p(x)+\varepsilon}\,I$: small near the data, large away from it. "Conformal" means a scalar times the identity: all directions cost the same. The provided `DiagonalMetric` is an anisotropic variant (Arvanitidis et al., 2016): $M(x)=\operatorname{diag}\bigl(\sum_n w_n(x)\,(x_n-x)^{2}+\rho\bigr)^{-1}$, with Gaussian weights $w_n(x)$ and the square taken per coordinate, which makes it cheap to move along the coordinate axes in which the nearby data is spread.

**Plots of a metric.** `lp.plot_metric(ax, metric, level=...)` shows a metric field on a slice of the workspace (a horizontal slice at height `level` seen from above, or a vertical slice at `x = level` seen from the front). The shading is the size of $M$, $\log_{10}(\det M)^{1/3}$: dark means expensive. The ellipses are the unit balls $\{\dot x:\dot x^\top M\dot x\le1\}$: a large ellipse means cheap motion, a long axis a cheap direction.

### Tasks
* **3.1** `ConformalMetric`: `density(x)` returns $\hat p(x)$, shape `(N,)`, and `metric(x)` (calling the object) returns $M(x)$, shape `(N, 3, 3)`. Use torch operations only, without loops over points, so that autograd can differentiate them (Part 4 needs it).

### The pouring data (provided)
The 20 training curves (blue: left lane, red: right lane) and the 2 held-out demos (black, dashed), in 3D, from above and from the front. All curves start at the same place.

In [ ]:
paths = data.load_pouring_paths()
X_train, V_train, A_train = (torch.as_tensor(a) for a in (paths.train_x, paths.train_v, paths.train_a))  # (20, 100, 3)
test = paths.test                                     # the held-out real demos: wine 2 (left), wine 4 (right)
train_real = paths.train_real[paths.train_real >= 0]  # 6 training curves that are recorded demos
eval_idx = np.r_[test, train_real]                    # the demos we evaluate on: 2 held-out, then 6 training
XLIM, YLIM, ZLIM = (0.05, 0.65), (-0.37, 0.37), (0.05, 0.32)   # the workspace, for plots
SIDE_COLOR = {"left": "C0", "right": "C3"}


def demo_style(i):
    """Style of real demo i: black if held out, else the colour of its side; solid for water, dashed for wine."""
    color = "k" if i in test else SIDE_COLOR["left" if paths.position[i] <= 2 else "right"]
    return dict(color=color, ls="-" if paths.liquid[i] == "water" else "--")


ax = plt.figure(figsize=(7, 6)).add_subplot(projection="3d")
for k in range(20):
    lp.plot_paths_3d(ax, paths.train_x[k], color=SIDE_COLOR[paths.train_side[k]], lw=1.2)
for i in test:
    lp.plot_paths_3d(ax, paths.x[i], color="k", lw=2.2, ls="--")
lp.set_equal_3d(ax, paths.train_x.reshape(-1, 3))
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.view_init(elev=25, azim=-60)
ax.set_title("Training curves and held-out demos (black)", fontsize=9)
plt.show()

axes = lp.path_axes(("xy", "yz"), title="Training set (colour by side) and held-out demos (black, dashed)")
for k in range(20):
    lp.plot_path(axes, paths.train_x[k], color=SIDE_COLOR[paths.train_side[k]], lw=1.2)
for i in test:
    lp.plot_path(axes, paths.x[i], color="k", lw=2, ls="--")
plt.show()

In [ ]:
class DiagonalMetric:
    """
    Provided. The diagonal density metric of Arvanitidis et al. (2016):
    M(x) = diag( sum_n w_n(x) (x_n - x)^2 + rho )^-1,  w_n(x) = exp(-|x - x_n|^2 / (2 sigma^2)),
    with the square taken per coordinate. data: positions (..., 3); x: tensor (N, 3).
    """

    def __init__(self, data, sigma=0.05, rho=1e-3):
        self.data = torch.as_tensor(np.asarray(data)).reshape(-1, 3)
        self.sigma, self.rho = sigma, rho

    def __call__(self, x):
        diff = self.data[None] - x[:, None, :]                          # (N, N_data, 3)
        w = torch.exp(-(diff**2).sum(-1) / (2 * self.sigma**2))         # (N, N_data)
        return torch.diag_embed(1.0 / ((w[..., None] * diff**2).sum(1) + self.rho))


class ConformalMetric:
    """
    The conformal density metric M(x) = I / (p(x) + eps), with the kernel density estimate
    p(x) = (1/N) sum_n exp(-|x - x_n|^2 / (2 sigma^2)) of the data positions x_n.
    data: positions (..., 3), NumPy or torch; x: tensor (N, 3).
    """

    def __init__(self, data, sigma=0.03, eps=0.01):
        self.data = torch.as_tensor(np.asarray(data)).reshape(-1, 3)
        self.sigma, self.eps = sigma, eps

    def density(self, x):
        """The kernel density estimate p(x), shape (N,)."""
        raise NotImplementedError

    def __call__(self, x):
        """The metric M(x), shape (N, 3, 3)."""
        raise NotImplementedError


checks.part3(ConformalMetric)

### Experiment 3.1: what the density metrics look like
The provided cell plots both metrics with `lp.plot_metric`: one row per metric, with a panel from above on the slice $z=0.2$ m and a panel from the front on the slice $x=0.4$ m, and the training curves on top.

In [ ]:
conformal = ConformalMetric(X_train, sigma=0.03, eps=0.01)
diagonal = DiagonalMetric(X_train, sigma=0.05, rho=1e-3)
for name, metric in [("conformal density metric", conformal), ("diagonal density metric", diagonal)]:
    axes = lp.path_axes(("xy", "yz"), title=name)
    lp.plot_metric(axes[0], metric, level=0.2, xlim=XLIM, ylim=YLIM)
    lp.plot_metric(axes[1], metric, level=0.4, xlim=YLIM, ylim=ZLIM)
    for k in range(20):
        lp.plot_path(axes, paths.train_x[k], color=SIDE_COLOR[paths.train_side[k]], lw=1)
    plt.show()

### Experiment 3.2: geodesics of the density metrics
The provided cells compute the geodesics between the end points of the demos `eval_idx` with `mt.evaluate(metric, paths, eval_idx)`, for the straight line (`mt.euclidean`), the conformal and the diagonal metric, and print the path errors (`error_table`). The figure shows the conformal metric (from above and from the front, as above) with the demos (thin) and their geodesics (thick), made by `plot_geodesics`. Parts 4 and 5 use the same table and the same figure.

→ Q3.1.

In [ ]:
def error_table(results):
    """Provided. results: {name: output of mt.evaluate(metric, paths, eval_idx)} -> path errors in mm."""
    rows = []
    for name, ev in results.items():
        e = ev["error_mm"]
        rows.append({"metric": name, "train (6 demos, mean)": e[2:].mean(),
                     **{f"held-out {paths.liquid[i]} {paths.position[i]}": e[k] for k, i in enumerate(test)}})
    return pd.DataFrame(rows).set_index("metric").round(1)


def plot_geodesics(metric, ev, title):
    """Provided. The metric (from above and from the front), the real demos (thin) and the geodesics of ev (thick)."""
    axes = lp.path_axes(("xy", "yz"), title=title)
    lp.plot_metric(axes[0], metric, level=0.2, xlim=XLIM, ylim=YLIM)
    lp.plot_metric(axes[1], metric, level=0.4, xlim=YLIM, ylim=ZLIM)
    for i, g in zip(eval_idx, ev["geodesics"]):
        lp.plot_path(axes, paths.x[i], lw=1, alpha=0.7, **demo_style(i))
        lp.plot_path(axes, g, lw=2.5, **demo_style(i))
    plt.show()

In [ ]:
results = {}
for name, metric in [("straight line", mt.euclidean), ("conformal", conformal), ("diagonal", diagonal)]:
    results[name] = mt.evaluate(metric, paths, eval_idx)
display(error_table(results))

In [ ]:
plot_geodesics(conformal, results["conformal"], "Conformal density metric: demos (thin) and geodesics (thick)")

### Questions, Part 3
**Q3.1** The conformal density metric does much better than the straight line, on the training and on the held-out demos. Explain why with your figures. What would its geodesic to the centre glass (position 3), where there is no data, look like?

**Answer Q3.1:**

_Your answer here._

**Checkpoint, Part 3.** Public checks pass; the metric figures, the error table and the geodesic figure; Q3.1 answered.

---
# Part 4: Learning a metric on the SPD manifold (34 points)

### Context
The density metric of Part 3 knows *where* the data is, not *how* the demonstrator moves. In this part we learn the second from the shape of the demos, and keep the first: the metric is the conformal density factor of Part 3 times a learned SPD field,
$$M_\theta(x)=c(x)\,A_\theta(x),\qquad c(x)=\frac{1}{\hat p(x)+\varepsilon},$$
so that moving stays cheap near the data and expensive away from it, while $A_\theta(x)$ decides which directions are cheap.

**The geodesic equation.** A path with constant speed is a geodesic of $M$ exactly when
$$\ddot x^k+\Gamma^k_{ij}(x)\,\dot x^i\dot x^j=0,\qquad \Gamma^k_{ij}=\tfrac12\,(M^{-1})_{kl}\bigl(\partial_iM_{lj}+\partial_jM_{li}-\partial_lM_{ij}\bigr),$$
with a sum over repeated indices and $\partial_i=\partial/\partial x^i$. The **Christoffel symbols** $\Gamma^k_{ij}(x)$ contain the first derivatives of $M$: a constant metric has $\Gamma=0$ and straight geodesics, and a varying metric bends them.

**The geodesic residual.** Instead of computing geodesics during training, which is expensive, we ask the training curves to satisfy the geodesic equation. At each sample of a curve, with $a=\ddot x+\Gamma(\dot x,\dot x)$,
$$r=L^2\,\frac{\lVert a_\perp\rVert^2}{\lVert\dot x\rVert^4},\qquad a_\perp=a-\frac{\langle a,\dot x\rangle}{\lVert\dot x\rVert^2}\,\dot x ,$$
where $L$ is the length of the curve. $r=0$ along a geodesic. Removing the component of $a$ along $\dot x$ makes $r$ independent of how fast the demonstrator moved: only the shape of the path counts. The factor $L^2$ makes $r$ dimensionless. For $M=I$, $r$ is the squared curvature of the path times $L^2$. `mt.geodesic_residual(metric, X, V, A, christoffel)` computes $r$.

**An SPD anchor field.** Place $K=12$ anchors at fixed centres $c_k$ (k-means of the training positions), give each an SPD matrix $P_k$, and blend them:
$$A_\theta(x)=\exp\Bigl(w_0(x)\log P_0+\sum_{k=1}^Kw_k(x)\log P_k\Bigr),\qquad w(x)=\operatorname{softmax}\Bigl(b,\ -\tfrac{\lVert x-c_1\rVert^2}{2h^2},\ \dots,\ -\tfrac{\lVert x-c_K\rVert^2}{2h^2}\Bigr).$$
The background $P_0=I$ is fixed and wins far from all anchors; $h$ is the kernel width. Blending the logarithms makes $A_\theta(x)$ SPD for every $x$. The parameters $\theta=(P_1,\dots,P_K)$ are points of the **SPD manifold** $\mathcal S^3_{++}$, and the loss is
$$\ell(\theta)=\operatorname{mean}(r)+\beta\sum_k d_{AI}(P_0,P_k)^2 ,$$
where the regulariser pulls the anchors towards the background.

**Riemannian gradient descent on $\mathcal S^n_{++}$.** Tangent vectors at $P$ are symmetric matrices. With the **affine-invariant metric** $\langle\xi,\eta\rangle_P=\operatorname{tr}(P^{-1}\xi P^{-1}\eta)$:
* the distance is $d_{AI}(P,Q)=\lVert\log(P^{-1/2}QP^{-1/2})\rVert_F$ (provided: `mt.spd_dist`);
* if $G=\partial\ell/\partial P$ is the Euclidean gradient (what autograd returns), the Riemannian gradient is $\operatorname{grad}\ell(P)=P\,\operatorname{sym}(G)\,P$, with $\operatorname{sym}(G)=\frac12(G+G^\top)$;
* the exponential map is $\operatorname{Exp}_P(\xi)=P^{1/2}\exp\bigl(P^{-1/2}\xi P^{-1/2}\bigr)P^{1/2}$, with the matrix exponential.

The step $P\leftarrow\operatorname{Exp}_P(-\eta\,\operatorname{grad}\ell(P))$ is the same as the Fréchet mean iteration of Part 1, on another manifold. It never leaves $\mathcal S^n_{++}$, and its size adapts to $P$: near a matrix with a small eigenvalue, the step in that direction is small too. We compare it with three **Euclidean workarounds**, called E1, E2 and E3, which keep the Euclidean gradient step of ordinary training but apply it to something else. They are implemented in `fit_anchor`, below: read its code to find out what each one optimises and how it obtains the anchors (Q4.1).

**Numerics.** `torch.linalg.eigh` returns infinite gradients when two eigenvalues are equal, for example at $P=I$, where training starts. The provided `mt.spd_sqrtm`, `mt.spd_invsqrtm` and `mt.spd_log` have safe gradients; use them, together with `torch.linalg.matrix_exp`.

### Tasks
* **4.1** `christoffel(metric, x)`: the Christoffel symbols as a tensor `G[n, k, i, j]` $=\Gamma^k_{ij}(x_n)$, shape `(N, 3, 3, 3)`. Get the derivatives $\partial_lM_{ij}$ with `torch.autograd.grad`, one call per entry $(i, j)$ (`M[:, i, j].sum()` gives the derivative at every point at once, because $M(x_n)$ depends only on $x_n$). Pass `create_graph=True`: training differentiates $\Gamma$ again, with respect to the parameters of the metric.
* **4.2** `airm_rgrad(P, G)`, `airm_exp(P, xi)` and `rgd_step(P, G, lr)`, for batches of matrices `(..., n, n)`.

In [ ]:
def christoffel(metric, x):
    """
    Christoffel symbols G[n, k, i, j] = Gamma^k_ij(x_n) of a metric field, shape (N, d, d, d),
    with the derivatives of M from torch.autograd.grad (create_graph=True).
    """
    x = x.detach().requires_grad_(True)
    M = metric(x)                                   # (N, d, d)
    N, d = x.shape
    dM = torch.zeros(N, d, d, d)                     # dM[n, l, i, j] = d_l M_ij at x_n
    raise NotImplementedError


def airm_rgrad(P, G):
    """Riemannian gradient P sym(G) P of the affine-invariant metric, from the Euclidean gradient G."""
    raise NotImplementedError


def airm_exp(P, xi):
    """Exp_P(xi) = P^1/2 expm(P^-1/2 xi P^-1/2) P^1/2 for SPD P and symmetric xi; symmetrise the result."""
    raise NotImplementedError


def rgd_step(P, G, lr):
    """One step of Riemannian gradient descent: Exp_P(-lr * grad), with grad = airm_rgrad(P, G)."""
    raise NotImplementedError


checks.part4(christoffel, airm_rgrad, airm_exp, rgd_step)

### The anchor field and the training loop (provided)
`AnchorField` implements $M_\theta(x)=c(x)A_\theta(x)$, with $c(x)I$ given by your `conformal` metric. `field(x, P)` evaluates it for anchors `P` of shape `(K, 3, 3)`, and `field.metric(P)` is the metric field for fixed anchors, to pass to `mt.evaluate` or `lp.plot_metric`. `fit_anchor` minimises the loss with one of the methods `"rgd"` (your `rgd_step`) and the Euclidean workarounds `"E1"`, `"E2"`, `"E3"` (→ Q4.1), and shows its progress with a bar. It stops when the loss is no longer finite or a matrix function fails, and returns the anchors and a history: the loss, the residual, the smallest eigenvalue of the anchors and the time per iteration, and `"failed"`, the iteration where it stopped (`None` if it did not). To save time, the training uses every fourth sample of the curves (`X4, V4, A4`, 25 per curve).

In [ ]:
from scipy.cluster.vq import kmeans2


class AnchorField:
    """M(x) = prior(x) @ expm( w_0(x) log P0 + sum_k w_k(x) log P_k ), w(x) = softmax(b, -|x - c_k|^2 / (2 h^2))."""

    def __init__(self, centres, h=0.25, b=-2.0, prior=None, P0=None):
        self.centres = torch.as_tensor(np.asarray(centres))      # (K, 3)
        self.h, self.b, self.prior = h, b, prior
        self.P0 = torch.eye(3) if P0 is None else torch.as_tensor(P0)
        self.log_P0 = mt.spd_log(self.P0)

    def weights(self, x):
        d2 = ((x[:, None, :] - self.centres[None]) ** 2).sum(-1)                              # (N, K)
        logits = torch.cat([torch.full_like(d2[:, :1], self.b), -d2 / (2 * self.h**2)], 1)  # (N, K + 1)
        return torch.softmax(logits, 1)

    def __call__(self, x, P):
        w = self.weights(x)
        S = w[:, :1, None] * self.log_P0 + torch.einsum("nk,kij->nij", w[:, 1:], mt.spd_log(P))
        A = torch.linalg.matrix_exp(S)
        return A if self.prior is None else self.prior(x) @ A

    def metric(self, P):
        P = P.detach().clone()
        return lambda x: self(x, P)


def fit_anchor(field, X, V, A, method="rgd", lr=0.3, beta=0.01, iters=200, eps=1e-3):
    """
    Minimise mean(r) + beta * sum_k d_AI(P0, P_k)^2 over the anchors, starting from P_k = P0.
    method: "rgd" (your rgd_step), "E1", "E2" or "E3". Returns (P, history).
    """
    K, P0 = len(field.centres), field.P0
    init = P0.expand(K, 3, 3).clone()
    if method in ("rgd", "E1"):
        theta = init.requires_grad_(True)
    elif method == "E2":
        theta = torch.linalg.cholesky(init).requires_grad_(True)
    elif method == "E3":
        theta = mt.spd_log(init).requires_grad_(True)
    to_P = {"E2": lambda t: t @ t.mT, "E3": lambda t: torch.linalg.matrix_exp(mt.sym(t))}.get(method, lambda t: t)
    hist = {"loss": [], "residual": [], "min_eig": [], "time": [], "failed": None}
    P_last, t0 = to_P(theta).detach().clone(), time.time()
    bar = tqdm(range(iters), desc=f"{method}, lr = {lr}")
    for it in bar:
        try:
            P = to_P(theta)
            r = mt.geodesic_residual(lambda x: field(x, P), X, V, A, christoffel).mean()
            loss = r + beta * (mt.spd_dist(P0, P) ** 2).sum()
            if not torch.isfinite(loss):
                raise FloatingPointError("the loss is not finite")
            G, = torch.autograd.grad(loss, theta)
            min_eig = torch.linalg.eigvalsh(P.detach()).min().item()
            with torch.no_grad():
                if method == "rgd":
                    theta.copy_(rgd_step(theta, G, lr))
                elif method == "E1":
                    lam, U = torch.linalg.eigh(mt.sym(theta - lr * G))
                    theta.copy_(U @ torch.diag_embed(lam.clamp_min(eps)) @ U.mT)
                else:
                    theta -= lr * G
        except NotImplementedError:
            raise
        except (FloatingPointError, RuntimeError) as e:     # RuntimeError: e.g. eigh fails on a broken matrix
            hist["failed"] = it
            print(f"{method}, lr = {lr}: stopped at iteration {it} ({str(e).splitlines()[0][:60]})")
            break
        P_last = P.detach().clone()
        hist["loss"].append(loss.item())
        hist["residual"].append(r.item())
        hist["min_eig"].append(min_eig)
        hist["time"].append(time.time() - t0)
        bar.set_postfix(loss=f"{loss.item():.4g}")
    bar.close()
    return P_last, hist


X4, V4, A4 = X_train[:, ::4], V_train[:, ::4], A_train[:, ::4]       # every fourth sample: (20, 25, 3)
centres, _ = kmeans2(X_train.reshape(-1, 3).numpy(), 12, seed=0, minit="++")
field = AnchorField(centres, h=0.25, b=-2.0, prior=conformal)

### Experiment 4.1: Riemannian gradient descent
The provided cells train the anchor field with your `rgd_step` (200 iterations, `lr=0.3`, β = 0.01, about 10 s), evaluate it on `eval_idx`, add it to the error table of Part 3, and show it with `plot_geodesics`. A second figure shows what was learned at the anchors alone: `lp.plot_anchors(axes, centres, P)` draws each $P_k$ as its unit ball at its centre (a long axis is a cheap direction), here over the training curves.

In [ ]:
P_rgd, hist_rgd = fit_anchor(field, X4, V4, A4, method="rgd", lr=0.3, beta=0.01)
results["anchor field"] = mt.evaluate(field.metric(P_rgd), paths, eval_idx)

print(f"loss {hist_rgd['loss'][0]:.2f} -> {hist_rgd['loss'][-1]:.2f}, residual {hist_rgd['residual'][-1]:.2f}, "
      f"time {hist_rgd['time'][-1]:.1f} s")
display(error_table(results))
axes = lp.path_axes(("xy", "yz"), title="The learned anchors P_k (unit balls) at their centres")
for k in range(20):
    lp.plot_path(axes, paths.train_x[k], color=SIDE_COLOR[paths.train_side[k]], lw=0.8, alpha=0.5)
lp.plot_anchors(axes, centres, P_rgd)
plt.show()

In [ ]:
plot_geodesics(field.metric(P_rgd), results["anchor field"], "Anchor field (RGD): demos (thin) and geodesics (thick)")

### Experiment 4.2: Riemannian versus Euclidean optimisation
The provided cells train E1, E2 and E3 with the same settings as RGD in Experiment 4.1 (`lr=0.3`). For each method, the table reports whether the run failed and where, the final loss, the smallest eigenvalue of the anchors during training, the number of **spikes** (iterations where the loss jumps up by more than 5%), the time, and, if the run finished, the held-out path errors. The figure shows the loss curves and the smallest eigenvalue of the anchors against the iteration.

→ Q4.2, Q4.3.

In [ ]:
runs = {"rgd": (P_rgd, hist_rgd)}
for method in ["E1", "E2", "E3"]:
    runs[method] = fit_anchor(field, X4, V4, A4, method=method, lr=0.3, beta=0.01)


def n_spikes(loss):
    loss = np.asarray(loss)
    return int((loss[1:] > 1.05 * loss[:-1]).sum())


rows = []
for method, (P, h) in runs.items():
    row = {"method": method, "failed at": h["failed"], "final loss": h["loss"][-1] if h["loss"] else np.nan,
           "min eig": min(h["min_eig"]) if h["min_eig"] else np.nan, "spikes": n_spikes(h["loss"]),
           "time [s]": h["time"][-1] if h["time"] else np.nan}
    if h["failed"] is None:
        ev = mt.evaluate(field.metric(P), paths, test)
        row.update({f"held-out {paths.liquid[i]} {paths.position[i]}": e for i, e in zip(test, ev["error_mm"])})
    rows.append(row)
display(pd.DataFrame(rows).set_index("method").round(3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 3.8))
lp.plot_curves({m: h["loss"] for m, (P, h) in runs.items()}, ax=axs[0], ylabel="loss")
lp.plot_curves({m: h["min_eig"] for m, (P, h) in runs.items()}, ax=axs[1], ylabel="smallest eigenvalue of the anchors")
plt.tight_layout(); plt.show()

### Experiment 4.3: what did each method learn?
For the four runs of Experiment 4.2, the provided cell plots the anchors of each method (`lp.plot_anchors`) over the training curves, from above and from the front, one row per method. For a run that failed, `fit_anchor` returned the anchors of its last finite iteration.

In [ ]:
for method in ["rgd", "E1", "E2", "E3"]:
    P, h = runs[method]
    axes = lp.path_axes(("xy", "yz"), figsize=(11, 3.6), title=f"{method}: learned anchors")
    for k in range(20):
        lp.plot_path(axes, paths.train_x[k], color=SIDE_COLOR[paths.train_side[k]], lw=0.8, alpha=0.5)
    lp.plot_anchors(axes, centres, P)
    plt.show()

### Questions, Part 4
**Q4.1** Read the code of `fit_anchor`. For each of E1, E2 and E3: which variable does the gradient step update, how are the anchors $P$ obtained from it, and are they guaranteed to be SPD? How does each differ from RGD?

**Answer Q4.1:**

_Your answer here._

**Q4.2** Describe how E1 and E2 fail in Experiment 4.2, with numbers from the table and the curves, and explain the failure from the update each one makes.

**Answer Q4.2:**

_Your answer here._

**Q4.3** The loss of E2 jumps up by orders of magnitude and then stays high. Use the curves of the smallest eigenvalue to explain what happens to the anchors before, at and after the jump.

**Answer Q4.3:**

_Your answer here._

**Checkpoint, Part 4.** Public checks pass; the learned field with its geodesics and anchors; the table and curves of Experiment 4.2; the anchor figures; Q4.1 to Q4.3 answered.

---
# Part 5: An SPD network for the metric (26 points)

### Context
The anchor field has a fixed structure: 12 anchors with a fixed width. A neural network can represent a much richer field. As in Part 4, the metric is the density factor times a learned SPD field, now the output of a network: $M(x)=c(x)\,S_\theta(x)$. The network output must be SPD for every input. Two designs:
* **A Cholesky head (V3).** The network outputs 6 numbers, which fill a lower-triangular $L(x)$ with a positive diagonal, and $S=LL^\top+\epsilon I$.
* **SPD layers** (Huang and Van Gool, 2017). Map the network output to the SPD manifold and continue there with layers that keep matrices SPD:
$$x\ \xrightarrow{\ \text{MLP}\ }\ v\in\mathbb R^{36}\ \xrightarrow{\ \text{vec\_to\_sym}\ }\ \xi\in\operatorname{Sym}_8\ \xrightarrow{\ \operatorname{Exp}_P\ }\ Y_1\in\mathcal S^8_{++}\ \xrightarrow{\ \text{BiMap}\ W_1\ }\ Y_2\in\mathcal S^6_{++}\ \xrightarrow{\ \text{ReEig}\ }\ Y_3\ \xrightarrow{\ \text{BiMap}\ W_2\ }\ S_\theta(x)\in\mathcal S^3_{++}$$
  * $\operatorname{Exp}_P$ is your `airm_exp`, at a **learned base point** $P\in\mathcal S^8_{++}$; `mt.vec_to_sym` divides the off-diagonal entries by $\sqrt2$, so that it preserves norms.
  * **BiMap** is $Y\mapsto W^\top YW$ with $W\in\mathbb R^{n\times m}$, $W^\top W=I$: $W$ lives on the **Stiefel manifold** $\operatorname{St}(n,m)$, and the output is SPD whenever $Y$ is. It is the SPD analogue of a linear layer.
  * **ReEig** acts on the eigenvalues, $Y=U\operatorname{diag}(\lambda)U^\top\mapsto U\operatorname{diag}\bigl(\epsilon+\operatorname{softplus}(\lambda-\epsilon)\bigr)U^\top$: a smooth version of clamping the eigenvalues at ε, the SPD analogue of a ReLU.

**Training on manifolds with geoopt.** The weights $P$, $W_1$, $W_2$ live on manifolds. geoopt handles this with two objects: a `geoopt.ManifoldParameter(tensor, manifold=...)` is a parameter that knows its manifold (here `geoopt.SymmetricPositiveDefinite()` or `geoopt.Stiefel()`), and `geoopt.optim.RiemannianAdam` is Adam that, for such parameters, converts the Euclidean gradient into a Riemannian one and moves along the manifold (with a retraction), as your `rgd_step` does; for ordinary parameters it is plain Adam. The rest of the training loop is the usual PyTorch one: `opt.zero_grad()`, compute the loss, `loss.backward()`, `opt.step()`. We compare three ways to train:
* **V1 Riemannian:** `ManifoldParameter`s and `RiemannianAdam`;
* **V2 naive:** plain tensors and Adam, ignoring the constraints;
* **V3:** the Cholesky network above, without SPD layers.

The loss is the residual of Part 4 plus a functional regulariser on the learned part, $\operatorname{mean}(r)+\gamma\,\operatorname{mean}_z\,d_{AI}(S_\theta(z),I)^2$, with $z$ running over the training positions and 200 random points in the box around them. The networks train on `X4, V4, A4`. Training one network takes 20–40 s.

### Tasks
* **5.1** `bimap(Y, W)` and `reeig(Y, eps)` for batches `(N, n, n)`.
* **5.2** In `SPDMetricNet.__init__`, for the variant `"riemannian"`: declare `self.P`, `self.W1`, `self.W2` as `geoopt.ManifoldParameter`s on the right manifolds, initialised with the tensors `P`, `W1`, `W2`.
* **5.3** `SPDMetricNet.shape(x)`: the chain above, $S_\theta(x)$. `forward(x)` (provided) multiplies it by the density factor.
* **5.4** In `fit_net`: create the optimiser (`RiemannianAdam` for the variant `"riemannian"`, `torch.optim.Adam` otherwise, learning rate `lr`), and write the optimisation step of the training loop.

In [ ]:
import geoopt


def bimap(Y, W):
    """BiMap layer: W^T Y W, for Y (N, n, n) and W (n, m); returns (N, m, m)."""
    raise NotImplementedError


def reeig(Y, eps=1e-2):
    """Smooth ReEig layer: U diag(eps + softplus(lambda - eps)) U^T for symmetric Y = U diag(lambda) U^T."""
    raise NotImplementedError


class SPDMetricNet(torch.nn.Module):
    """
    M(x) = prior(x) @ S(x), S(x) = BiMap W2 (ReEig (BiMap W1 (Exp_P (vec_to_sym (MLP(x)))))), SPD 3 x 3.
    variant: "riemannian" (P, W1, W2 are geoopt ManifoldParameters) or "euclidean" (plain tensors).
    prior: a metric field (here the conformal density metric) or None.
    The input of the MLP is normalised as (x - centre) / scale.
    """

    def __init__(self, variant="riemannian", prior=None, n=8, hidden=64, eps=1e-2, centre=(0.0, 0.0, 0.0), scale=1.0, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        torch.manual_seed(seed)
        self.variant, self.prior, self.eps, self.n = variant, prior, eps, n
        self.centre, self.scale = torch.as_tensor(centre), scale
        self.mlp = torch.nn.Sequential(torch.nn.Linear(3, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, hidden),
                                       torch.nn.Tanh(), torch.nn.Linear(hidden, self.n * (self.n + 1) // 2))
        with torch.no_grad():                       # start close to S(x) = W2^T W1^T P W1 W2 = I
            self.mlp[-1].weight.mul_(0.1)
            self.mlp[-1].bias.zero_()
        P = torch.eye(self.n)
        W1 = torch.linalg.qr(torch.randn(self.n, 6, generator=g))[0]
        W2 = torch.linalg.qr(torch.randn(6, 3, generator=g))[0]
        if variant == "riemannian":
            # TODO 5.2: self.P, self.W1, self.W2 as geoopt.ManifoldParameter on the right manifolds
            raise NotImplementedError("declare P, W1, W2 as geoopt.ManifoldParameter")
        elif variant == "euclidean":
            self.P, self.W1, self.W2 = (torch.nn.Parameter(t) for t in (P, W1, W2))

    def weights(self):
        """The base point P and the BiMap weights W1, W2."""
        return self.P, self.W1, self.W2

    def shape(self, x):
        """S(x), shape (N, 3, 3). Use self.mlp, mt.vec_to_sym, airm_exp, bimap, reeig (with eps=self.eps)."""
        P, W1, W2 = self.weights()
        v = self.mlp((x - self.centre) / self.scale)
        raise NotImplementedError

    def forward(self, x):
        S = self.shape(x)
        return S if self.prior is None else self.prior(x) @ S


checks.part5(bimap, reeig, SPDMetricNet, airm_exp)

### The Cholesky network and the training loop
`CholeskyNet` is V3 (provided). `fit_net` trains a network (task 5.4) and shows its progress with a bar. Besides the loss and the residual, it records the **worst sample** at each iteration (the largest residual, and which curve and sample it belongs to), and the largest violation of the constraints, $\max|W^\top W-I|$ and the smallest eigenvalue of $P$.

In [ ]:
class CholeskyNet(torch.nn.Module):
    """V3: M(x) = prior(x) @ S(x), S = L L^T + eps I, with L lower-triangular from the 6 outputs of an MLP."""

    def __init__(self, prior=None, hidden=64, eps=1e-2, centre=(0.0, 0.0, 0.0), scale=1.0, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.prior, self.eps, self.centre, self.scale = prior, eps, torch.as_tensor(centre), scale
        self.mlp = torch.nn.Sequential(torch.nn.Linear(3, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, hidden),
                                       torch.nn.Tanh(), torch.nn.Linear(hidden, 6))
        with torch.no_grad():
            self.mlp[-1].weight.mul_(0.1)
            self.mlp[-1].bias.zero_()

    def shape(self, x):
        o = self.mlp((x - self.centre) / self.scale)
        diag = torch.nn.functional.softplus(o[:, :3] + np.log(np.e - 1))         # = 1 at initialisation
        L = torch.diag_embed(diag) + mt.vec_to_sym(torch.cat([torch.zeros_like(diag), o[:, 3:]], 1), 3).tril(-1)
        return L @ L.mT + self.eps * torch.eye(3)

    def forward(self, x):
        S = self.shape(x)
        return S if self.prior is None else self.prior(x) @ S


def fit_net(model, X, V, A, iters=300, lr=3e-3, gamma=0.1, n_box=200, seed=0, desc=None):
    """
    Minimise mean(r) + gamma * mean_z d_AI(S(z), I)^2. Returns a history with the loss, the residual,
    the worst sample (r, curve, sample), the time and, for SPDMetricNet, the constraint violations max |W^T W - I| and the smallest
    eigenvalue of P.
    """
    # TODO 5.4: opt = RiemannianAdam for the variant "riemannian", torch.optim.Adam otherwise, over model.parameters()
    raise NotImplementedError("create the optimiser")
    g = torch.Generator().manual_seed(seed)
    pts = X.reshape(-1, 3)
    lo, hi = pts.min(0).values - 0.05, pts.max(0).values + 0.05
    hist = {"loss": [], "residual": [], "worst": [], "time": [], "stiefel": [], "min_eig": []}
    t0 = time.time()
    desc = desc or f"{type(model).__name__} ({getattr(model, 'variant', 'cholesky')})"
    bar = tqdm(range(iters), desc=desc)
    for it in bar:
        # TODO 5.4: reset the gradients
        rr = mt.geodesic_residual(model, X, V, A, christoffel)      # (D, T)
        r = rr.mean()
        Z = torch.cat([pts, lo + (hi - lo) * torch.rand(n_box, 3, generator=g)])
        S = model.shape(Z)
        loss = r + gamma * (mt.spd_dist(torch.eye(3), S) ** 2).mean()
        if not torch.isfinite(loss):
            print(f"the loss is not finite at iteration {it}, stopping")
            break
        # TODO 5.4: backpropagate and take an optimiser step
        hist["loss"].append(loss.item())
        hist["residual"].append(r.item())
        hist["worst"].append((rr.max().item(), *divmod(int(rr.argmax()), rr.shape[1])))   # (r, curve, sample)
        hist["time"].append(time.time() - t0)
        bar.set_postfix(loss=f"{loss.item():.4g}", residual=f"{r.item():.3g}")
        if isinstance(model, SPDMetricNet):
            with torch.no_grad():
                P, W1, W2 = model.weights()
                hist["stiefel"].append(max((W.mT @ W - torch.eye(W.shape[1])).abs().max().item() for W in (W1, W2)))
                hist["min_eig"].append(torch.linalg.eigvalsh(P).min().item())
    bar.close()
    return hist


NET = dict(prior=conformal, centre=X_train.reshape(-1, 3).mean(0), scale=0.1)   # density factor, input normalisation

### Experiment 5.1: the Riemannian SPD network
The provided cells train V1 (`SPDMetricNet(**NET)` with your `fit_net` on `X4, V4, A4`), evaluate it on `eval_idx`, add it to the error table, and show it with `plot_geodesics`.

In [ ]:
net_v1 = SPDMetricNet(**NET)
hist_v1 = fit_net(net_v1, X4, V4, A4)
results["SPD net (V1)"] = mt.evaluate(net_v1, paths, eval_idx)

In [ ]:
print(f"residual {hist_v1['residual'][-1]:.2f}, time {hist_v1['time'][-1]:.0f} s")
display(error_table(results))

In [ ]:
plot_geodesics(net_v1, results["SPD net (V1)"], "SPD network (V1): demos (thin) and geodesics (thick)")

### Experiment 5.2: three ways to train
The provided cells train V2 (`variant="euclidean"`) and V3 (`CholeskyNet(**NET)`) like V1, evaluate them on `eval_idx`, and show the table with the constraint violations and the spikes. The figure shows the three loss curves and, in a second panel, the residual of the worst sample against the iteration; below it, the worst sample at each spike is listed. Finally, each network is shown with `plot_geodesics`.

→ Q5.1, Q5.2.

In [ ]:
nets = {"V1": (net_v1, hist_v1)}

for name, make in [("V2", lambda: SPDMetricNet(variant="euclidean", **NET)),
                   ("V3", lambda: CholeskyNet(**NET))]:
    net = make()
    nets[name] = (net, fit_net(net, X4, V4, A4))

for name in ["V2", "V3"]:
    results[f"SPD net ({name})"] = mt.evaluate(nets[name][0], paths, eval_idx)

rows = []
for name, (net, h) in nets.items():
    e = results[f"SPD net ({name})"]["error_mm"]
    rows.append({"variant": name, "residual": h["residual"][-1], "train [mm]": e[2:].mean(), "held-out [mm]": e[:2].mean(),
                 "max |WᵀW - I|": max(h["stiefel"]) if h["stiefel"] else np.nan,
                 "min eig P": min(h["min_eig"]) if h["min_eig"] else np.nan,
                 "spikes": n_spikes(h["loss"]), "time [s]": h["time"][-1]})
display(pd.DataFrame(rows).set_index("variant").round(3))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 3.8))
lp.plot_curves({name: h["loss"] for name, (net, h) in nets.items()}, ax=axs[0], ylabel="loss")
lp.plot_curves({name: [w[0] for w in h["worst"]] for name, (net, h) in nets.items()}, ax=axs[1],
               ylabel="residual of the worst sample")
plt.show()

for name, (net, h) in nets.items():
    L = np.asarray(h["loss"])
    spikes = np.flatnonzero(L[1:] > 1.05 * L[:-1]) + 1
    print(f"{name}: {len(spikes)} spikes; worst sample (curve, sample) at the spikes:",
          [h["worst"][i][1:] for i in spikes][:12])

In [ ]:
for name in ["V2", "V3"]:
    plot_geodesics(nets[name][0], results[f"SPD net ({name})"], f"{name}: demos (thin) and geodesics (thick)")

### Questions, Part 5
**Q5.1** Compare V1 and V2 (Experiment 5.2). Which one keeps $P$ SPD and $W$ on the Stiefel manifold, and does it matter for the result here? What could go wrong with V2 with a larger learning rate or a longer training?

**Answer Q5.1:**

_Your answer here._

**Q5.2** Some loss curves have spikes. Find out where they come from, using what `fit_net` recorded about the worst sample at each iteration. Explain the mechanism.

**Answer Q5.2:**

_Your answer here._

**Checkpoint, Part 5.** Public checks pass; the V1 figures; the table of Experiment 5.2, the loss curves and the metric figures; Q5.1 and Q5.2 answered.